In [1]:
import sys
sys.path.append("..")

In [2]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [3]:
def append_result(d, objective, loss, cost, m1_validity, wc_validity, m1_expectation, wc_expectation):
    d['cost'].append(cost)
    d['m1_validity'].append(m1_validity)
    d['wc_validity'].append(wc_validity)
    d['m1_probability'].append(m1_expectation)
    d['wc_probability'].append(wc_expectation)
    d['loss'].append(loss)
    d['J'].append(objective) 
    
def get_result(d, algorithm, seed, alpha, lamb, theta_0, theta_r):
    result = {
        'algorithm': algorithm, 
        'seed': seed,
        'alpha': alpha,
        'lambda': lamb,
        'theta_0': theta_0,
        'theta_r': theta_r
        }
    
    for key in d.keys():
        result[key] = np.mean(d[key])
    return result

In [9]:
def get_model_adv_pga_linf(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(X_0).float()
    X_r = torch.tensor(X_r).float()
    weights_min = [cfr.model[0].weight.detach().clone()-alpha, cfr.model[2].weight.detach().clone()-alpha, cfr.model[4].weight.detach().clone()-alpha, cfr.model[6].weight.detach().clone()-alpha]
    weights_max = [cfr.model[0].weight.detach().clone()+alpha, cfr.model[2].weight.detach().clone()+alpha, cfr.model[4].weight.detach().clone()+alpha, cfr.model[6].weight.detach().clone()+alpha]
    bias_min = [cfr.model[0].bias.detach().clone()-alpha, cfr.model[2].bias.detach().clone()-alpha, cfr.model[4].bias.detach().clone()-alpha, cfr.model[6].bias.detach().clone()-alpha]
    bias_max = [cfr.model[0].bias.detach().clone()+alpha, cfr.model[2].bias.detach().clone()+alpha, cfr.model[4].bias.detach().clone()+alpha, cfr.model[6].bias.detach().clone()+alpha]
    
    cfr_adv = deepcopy(cfr)

    # NEW CODE
    for module in cfr_adv.model.children():
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)

    # optimizer = optim.Adam(cfr_adv.model.parameters(), maximize=True)
    optimizer = optim.SGD(cfr_adv.model.parameters(), lr=1, maximize=True)
    loss_fn = torch.nn.BCELoss(reduction='mean')

    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    # for epoch in range(pga_max_iter):
    while loss_diff > 1e-4:
        if i == pga_max_iter:
            break

        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv.model(X_r)
        y_target = torch.ones(f_x.shape).to(torch.float32)
        bce_loss = loss_fn(f_x, y_target)
        loss = bce_loss
        
        loss.backward()
        optimizer.step()
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
        
        with torch.no_grad():
            # clamp model parameters to -alpha, alpha range
            cfr_adv.model[0].weight.clamp_(weights_min[0], weights_max[0])
            cfr_adv.model[2].weight.clamp_(weights_min[1], weights_max[1])
            cfr_adv.model[4].weight.clamp_(weights_min[2], weights_max[2])
            cfr_adv.model[6].weight.clamp_(weights_min[3], weights_max[3])
            cfr_adv.model[0].bias.clamp_(bias_min[0], bias_max[0])
            cfr_adv.model[2].bias.clamp_(bias_min[1], bias_max[1])
            cfr_adv.model[4].bias.clamp_(bias_min[2], bias_max[2])
            cfr_adv.model[6].bias.clamp_(bias_min[3], bias_max[3])

    return cfr_adv

In [10]:
def project_l1_ball(x: torch.Tensor, eps: float) -> torch.Tensor:
    """
    Project x onto the L1-ball {z : ||z||_1 <= eps}.
    Works for any shape. Preserves device/dtype.
    """
    # Flatten
    orig_shape = x.shape
    v = x.detach().reshape(-1)

    # Already feasible
    if torch.linalg.norm(v, ord=1) <= eps:
        return x

    # Sort |v| descending
    u, _ = torch.sort(v.abs(), descending=True)
    sv = torch.cumsum(u, dim=0)

    j = torch.arange(1, u.numel() + 1, device=v.device, dtype=v.dtype)
    # Find rho = max { j : u_j > (sv_j - eps)/j }
    cond = u * j > (sv - eps)
    rho = torch.nonzero(cond, as_tuple=False).max()
    theta = (sv[rho] - eps) / (rho.to(v.dtype) + 1)

    # Soft-threshold with theta and reshape back
    w = torch.sign(v) * torch.clamp(v.abs() - theta, min=0)
    return w.view(orig_shape)

def get_model_adv_pga_l1(X_0, X_r, cfr, alpha, lamb, pga_max_iter: int = 100):
    X_0 = torch.tensor(X_0).float()
    X_r = torch.tensor(X_r).float()
    
    cfr_adv = deepcopy(cfr)

    # NEW CODE
    for module in cfr_adv.model.children():
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.constant_(module.bias, 0.01)

    optimizer = optim.SGD(cfr_adv.model.parameters(), lr=0.1, maximize=True)
    loss_fn = torch.nn.BCELoss(reduction='mean')

    loss = torch.tensor(1.)
    loss_diff = 1
    i = 0
    while loss_diff > 1e-4:
    # for epoch in range(pga_max_iter):
        if i == pga_max_iter:
            break
        
        prev_loss = loss.clone().detach()
        optimizer.zero_grad()
        
        f_x = cfr_adv.model(X_r)
        y_target = torch.ones(f_x.shape).to(torch.float32)
        bce_loss = loss_fn(f_x, y_target)
        loss = bce_loss
        
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            for param_adv, param in zip(cfr_adv.model.parameters(), cfr.model.parameters()):
                delta = param_adv - param
                delta_proj = project_l1_ball(delta, alpha)
                param_adv.copy_(param + delta_proj)
        
        loss_diff = torch.dist(prev_loss, loss, 1)
        i += 1
    return cfr_adv

In [11]:
def evaluate_performance_many_nn(X_0, X_r, alpha, lamb, seed, alg, theta_adv_method='Linf', dataset=None):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}

    cfr = NN(X_0.shape[1])
    cfr.model.load_state_dict(torch.load(f"../results/recourse_model/{dataset}_{seed}.pth"))       
    n = len(X_r)

    for i in tqdm.trange(n, desc=f'[{alg.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        if theta_adv_method=='L-1':
            cfr_adv = get_model_adv_pga_l1(x_0, x_r, cfr, alpha, lamb, 99)
        else:
            cfr_adv = get_model_adv_pga_linf(x_0, x_r, cfr, alpha, lamb, 99)
        J = RecourseCost(x_0, lamb)
        
        bce_loss_opt, cost_opt, rob_opt = J.eval_nonlinear(x_r.reshape((1,len(x_r))), cfr_adv.model, True)
        m1_validity_opt = cfr.predict(x_r.reshape(1,-1))[0]
        m1_expectation_opt = cfr.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity_opt = cfr_adv.predict(x_r.reshape(1,-1))[0]
        wc_expectation_opt = cfr_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, rob_opt, bce_loss_opt, cost_opt, m1_validity_opt, wc_validity_opt, m1_expectation_opt, wc_expectation_opt)
        
    return get_result(results, alg, seed, alpha, lamb, None, None)

In [4]:
def get_theta_adv_linf(X_0, X_r, theta_0, alpha, lamb):
    
    theta_adv = deepcopy(theta_0)
    
    for i in range(theta_0.shape[0]):
        theta_r_min = deepcopy(theta_0)
        theta_r_max = deepcopy(theta_0)
        
        theta_r_min[i] -= alpha
        theta_r_max[i] += alpha
        weights_r_min, bias_r_min = theta_r_min[:-1], theta_r_min[[-1]]
        weights_r_max, bias_r_max = theta_r_max[:-1], theta_r_max[[-1]]
        J_min, J_max = [], []
        for xi in range(len(X_r)):
            x_0 = X_0[xi]
            x_r = X_r[xi]
            J = RecourseCost(x_0, lamb)
            j_min = J.eval(x_r, weights_r_min, bias_r_min)
            j_max = J.eval(x_r, weights_r_max, bias_r_max)
            J_min.append(j_min)
            J_max.append(j_max)
        
        
        if np.mean(J_min) > np.mean(J_max):
            theta_adv[i] -= alpha
        else:
            theta_adv[i] += alpha

    return theta_adv

In [5]:
def get_theta_adv_l1(X_0, X_r, theta_0, alpha, lamb):
    thetas = generateThetas(theta_0, alpha)
    if alpha == 0:
        return thetas[0].copy()
    
    Js = np.empty((X_0.shape[0], thetas.shape[0]))
    for i in range(X_0.shape[0]):
         J = RecourseCost(X_0[i], lamb)
         for j, theta in enumerate(thetas):  
            Js[i, j] = J.eval(X_r[i], theta[:-1], np.array([theta[-1]]))
    
    Js_sum = Js.sum(axis=0) 
    Js_sum_maxI = np.argmax(Js_sum)
    theta_adv = thetas[Js_sum_maxI]

    return theta_adv

def generateThetas(theta0 : np.ndarray, alpha):
        # theta0 has bias
        thetas = theta0.copy()
        if alpha == 0:
            return np.array([thetas])
        
        thetas = np.repeat(thetas.reshape(1, theta0.size), (theta0.size * 2) - 1, axis=0)
        thetas_i = 0

        for i in range(theta0.size):
            if i == theta0.size - 1:
                thetas[thetas_i][i] -= alpha
                thetas_i += 1
                break

            thetas[thetas_i][i] += alpha
            thetas_i += 1
            thetas[thetas_i][i] -= alpha
            thetas_i += 1

        return thetas

In [6]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    # xP has bias
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, bias_adv))

def calTheta(xP: np.array, weights: np.array, bias: np.array, alpha: float, methods: str):
    if methods == "L-inf":
        thetaP = calThetaAdv_linf(xP, weights, bias, alpha)
    else:
        thetaP = calThetaAdv_l1(np.hstack((xP, np.array([1]))), np.hstack((weights, bias)), alpha)

    return thetaP[:-1], np.array([thetaP[-1]])

In [7]:
def evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method='L-inf'):
    results = {'cost': [], 'm1_validity': [], 'wc_validity': [], 'm1_probability': [], 'wc_probability': [], 'loss': [], 'J': []}
    
    n = len(X_r)
    Y_0 = np.hstack((np.ones((n//2,)), np.zeros((n - n//2,))))
    
    clf = LR()
    clf.train(X_0, Y_0)
    clf_adv = deepcopy(clf)


    for i in tqdm.trange(n, desc=f'[] [] [{algorithm.capitalize()}] [ seed={seed} ] [ α={alpha} ] [ λ={lamb} ]', colour='#0091ff'):
        x_0 = X_0[i]
        x_r = X_r[i]
        t_0 = theta_0[i]
        w_0, b_0 = t_0[:-1], t_0[[-1]]
        if alpha != 0:
            w_0_adv, b_0_adv = calTheta(x_r, w_0, b_0, alpha, theta_adv_method)
        else:
            w_0_adv, b_0_adv = w_0.copy(), b_0.copy()

        clf.model.coef_ = w_0.reshape(1,-1)
        clf.model.intercept_ = b_0
        clf_adv.model.coef_ = w_0_adv.reshape(1,-1)
        clf_adv.model.intercept_ = b_0_adv

        J = RecourseCost(x_0, lamb)
        bce_loss, cost, price = J.eval(x_r, w_0_adv, b_0_adv, True)

        m1_validity = clf.predict(x_r.reshape(1,-1))[0]
        m1_probability = clf.predict_proba(x_r.reshape(1,-1))[0,1]
        
        wc_validity = clf_adv.predict(x_r.reshape(1,-1))[0]
        wc_probability = clf_adv.predict_proba(x_r.reshape(1,-1))[0,1]
        
        append_result(results, price, bce_loss, cost, m1_validity, wc_validity, m1_probability, wc_probability)
        
    return get_result(results, algorithm, seed, alpha, lamb, None, None)

In [106]:
def runCostValidityTradeoff (results: dict, params: dict):
    for model in params['base_model']:
        for dataset in params['data']:
            model_dataset_name = model + '_' + dataset

            if model == "lr" and dataset == "sba":
                continue

            for algorithm in params['algorithms']:
                for seed in params['seeds']:
                    for v_alpha in params['alphas'][algorithm]:
                        for v_lamb in params['lambdas'][model_dataset_name][algorithm]:
                            data = pd.read_pickle(f"../results/recourse/{model}_{dataset}_{algorithm}_{v_lamb}_{v_alpha}_{seed}.pkl")
                            alpha = data["alpha"].unique().item()
                            lamb = data["lambda"].unique().item()
                            X_0 = np.stack(data["x_0"])
                            X_r = np.stack(data["x_r"])
                            theta_0 = np.stack(data["theta_0"])

                            if params['include_mask'] and algorithm != "L1PSD":
                                data_l1psd = pd.read_pickle(f"../results/recourse/{model}_{dataset}_L1PSD_0.1_0.1_{seed}.pkl")
                                mask_i = data_l1psd["i"].to_numpy()
                                X_0 = X_0[mask_i]
                                X_r = X_r[mask_i]
                                theta_0 = theta_0[mask_i]
                            else:
                                data_l1psd = pd.read_pickle(f"../results/recourse/{model}_{dataset}_L1PSD_0.1_0.1_{seed}.pkl")
                                mask_i = data_l1psd["i"].to_numpy()
                                if dataset == "sba" and model == "lr" and v_alpha == 0.5:
                                    data = data[data['i'].isin(mask_i)]    
                                elif dataset == "sba" and model == "nn" and v_alpha == 0.5:
                                    data = data[data['i'].isin(mask_i)]

                                X_0 = np.stack(data["x_0"])
                                X_r = np.stack(data["x_r"])
                                theta_0 = np.stack(data["theta_0"])

                            if model == "lr":
                                res = evaluate_performance_each(X_0, X_r, theta_0, alpha, lamb, seed, algorithm, theta_adv_method="L-1" if "L1" in algorithm else "L-inf")
                            elif model == "nn":
                                res = evaluate_performance_many_nn(X_0, X_r, alpha, lamb, seed, algorithm, "L-1" if "L1" in algorithm else "L-inf", dataset=dataset)
                            
                            
                            results['model'].append(model)
                            results['dataset'].append(dataset)
                            results['algorithm'].append(algorithm)
                            results['seed'].append(seed)
                            results['alpha'].append(alpha)
                            results['lambda'].append(lamb)
                            results['Cost'].append(res['cost'])
                            results['Current Validity'].append(res['m1_probability'])
                            results['Worst Case Validity'].append(res['wc_probability'])
                            results['BCE Loss'].append(res['loss'])
                            results['J'].append(res['J'])
    
    df_results = pd.DataFrame(results)
    return df_results

In [116]:
params = {}
# 'lr', 'nn'
params['base_model'] = ['lr', 'nn']
# 'synthetic', 'german', 'sba'
params['data'] = ['german', 'sba']
params['seeds'] = range(5)
# 'Alg1', 'L1PSD', 'ROARLInf', 'ROARL1'
params['algorithms'] = ['Alg1', 'L1PSD', 'ROARLInf', 'ROARL1']
# 'ONE', 'MANY', 'LARGESTALPHA', 'SMALLESTALPHA', 'THETA0'
params['adv_method'] = 'ONE'
params['include_base_model'] = False
params['include_mask'] = True

params['alphas'] = {params['algorithms'][0] : [0.5],
            params['algorithms'][1] : [0.5], 
            params['algorithms'][2] : [0.5], 
            params['algorithms'][3] : [0.5]}

params['lambdas'] = dict()
# lr_german
# params['lambdas']['lr_german'] = {params['algorithms'][0] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1))).round(5),
#             params['algorithms'][1] : [0.5,0.3,0.1,0.04,0.01,0.004,0.001], 
#             params['algorithms'][2] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7), 
#             params['algorithms'][3] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,0.55, 0.1), np.array([0.0001, 0.00001]))).round(7)}
params['lambdas']['lr_german'] = {params['algorithms'][0] : [0.1],
            params['algorithms'][1] : [0.1], 
            params['algorithms'][2] : [0.1], 
            params['algorithms'][3] : [0.1]}

# nn_german
# params['lambdas']['nn_german'] = {params['algorithms'][0] : np.hstack((np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1), np.array([2.0, 3.0, 4.0, 5.0]))).round(5),
#             params['algorithms'][1] : [3.0, 0.7, 0.3, 0.1, 0.05, 0.01, 0.001], 
#             params['algorithms'][2] : np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9), 
#             params['algorithms'][3] : np.hstack((np.array([1e-7,1e-6,1e-5,1e-4]),np.arange(0.001,0.0105,0.001),np.arange(0.02, 0.105, 0.01),np.arange(0.2,1.05, 0.1))).round(9)}
params['lambdas']['nn_german'] = {params['algorithms'][0] : [0.1],
            params['algorithms'][1] : [0.1], 
            params['algorithms'][2] : [0.1], 
            params['algorithms'][3] : [0.1]}

# lr_sba
# params['lambdas']['lr_sba'] = {params['algorithms'][0] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.3, 2.6, 2.8, 3.0, 3.1, 3.3, 3.5],
#             params['algorithms'][1] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.0, 3.5], 
#             params['algorithms'][2] : [0.001, 0.01, 0.08, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5], 
#             params['algorithms'][3] : [0.001, 0.01, 0.08,0.1, 0.3, 0.5, 0.7, 0.9, 1.0, 1.1, 1.4, 1.5, 1.6, 1.8, 2, 2.1, 2.8, 3.5]}
params['lambdas']['lr_sba'] = {params['algorithms'][0] : [0.1],
            params['algorithms'][1] : [0.1], 
            params['algorithms'][2] : [0.1], 
            params['algorithms'][3] : [0.1]}

# nn_sba
# params['lambdas']['nn_sba'] = {params['algorithms'][0] :[0.00001, 0.0001, 0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5],
#             params['algorithms'][1] : [0.001, 0.01, 0.1, 0.7, 1.4, 2.1, 2.8, 3.5], 
#             params['algorithms'][2] : [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5], 
#             params['algorithms'][3] : [0.001, 0.01, 0.1, 0.3, 0.5, 0.7, 1.4, 1.7, 1.9, 2.1, 2.8, 3.5]}
params['lambdas']['nn_sba'] = {params['algorithms'][0] : [0.1],
            params['algorithms'][1] : [0.1], 
            params['algorithms'][2] : [0.1], 
            params['algorithms'][3] : [0.1]}

results = {
    'model': [],
    'dataset': [],
    'algorithm': [],
    'seed': [],
    'alpha': [],
    'lambda': [],
    'Cost': [],
    'Current Validity': [],
    'Worst Case Validity': [],
    'BCE Loss': [],
    'J': []
}

df_results = runCostValidityTradeoff(results, params)

[] [] [Alg1] [ seed=0 ] [ α=0.5 ] [ λ=0.1 ]: 100%|██████████| 16/16 [00:00<00:00, 1152.80it/s]


[Roarl1] [ seed=4 ] [ α=0.5 ] [ λ=0.1 ]: 100%|██████████| 6/6 [00:00<00:00,  9.04it/s]


# Two Lambda - All Metric

In [117]:
df_results_sampled = df_results[df_results['lambda'].isin([0.1])]
df_results_sampled

,model,dataset,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,lr,german,Alg1,0,0.5,0.1,4.692981,0.714100,5.345221e-01,0.627491,1.096789
1,lr,german,Alg1,1,0.5,0.1,5.929775,0.695424,5.806884e-01,0.543541,1.136518
2,lr,german,Alg1,2,0.5,0.1,5.584564,0.708348,5.425054e-01,0.611634,1.170090
3,lr,german,Alg1,3,0.5,0.1,4.521147,0.700077,5.463553e-01,0.605098,1.057213
4,lr,german,Alg1,4,0.5,0.1,6.053436,0.724320,5.724643e-01,0.558570,1.163913
5,lr,german,L1PSD,0,0.5,0.1,4.811212,0.822745,7.382504e-01,0.304404,0.785525
6,lr,german,L1PSD,1,0.5,0.1,4.491083,0.716880,6.062849e-01,0.502228,0.951336
7,lr,german,L1PSD,2,0.5,0.1,5.177591,0.809899,7.105335e-01,0.342518,0.860277
8,lr,german,L1PSD,3,0.5,0.1,4.480200,0.842408,7.564594e-01,0.279723,0.727743
9,lr,german,L1PSD,4,0.5,0.1,5.721493,0.809252,7.072300e-01,0.346641,0.918790


In [118]:
df_results_sampled['lambda'] = df_results_sampled['lambda'].astype(str)
df_results_sampled['model'] = df_results_sampled['model'].replace(['lr', 'nn'], ['LR', 'NN'])
df_results_sampled['dataset'] = df_results_sampled['dataset'].replace(['german', 'sba'], ['German', 'SBA'])

pvt = df_results_sampled.pivot_table(values=['Current Validity', 'Worst Case Validity','Cost','J'], index=['model', 'algorithm'], columns=['dataset', 'lambda'], aggfunc=["mean", "std"])

pvt = pvt.rename(index={'Alg1': 'LInf'}, level='algorithm')
pvt.index.set_names(["Model", "Algorithm"], inplace=True)
pvt.columns.set_names(['AggFunc', 'Metric', 'Dataset', 'Lambda'], inplace=True)
pvt.columns = pvt.columns.reorder_levels(['AggFunc', 'Dataset', 'Lambda', 'Metric'])
pvt = pvt.sort_index(axis=1)

lambda_order = ["0.1", "0.01", "0.001"]
metric_order = ["Current Validity", "Worst Case Validity", "Cost", "J"]
algorithm_order = ['LInf', 'ROARLInf', 'L1PSD', 'ROARL1']
pvt = pvt.reindex(columns=lambda_order, level="Lambda").reindex(columns=metric_order, level= "Metric").reindex(index=algorithm_order, level="Algorithm")

pvt.style


In [119]:
new_pvt = pvt['mean'].round(2).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

In [120]:
new_pvt.style

In [121]:
lr_linf_pct = ((pvt.loc['LR', 'ROARLInf'] - pvt.loc['LR', 'LInf']) * 100 / pvt.loc['LR', 'LInf']).round(2)
lr_linf_pct = lr_linf_pct.loc['mean'].astype(str)
lr_l1_pct = ((pvt.loc['LR', 'ROARL1'] - pvt.loc['LR', 'L1PSD']) * 100 / pvt.loc['LR', 'L1PSD']).round(2)
lr_l1_pct = lr_l1_pct.loc['mean'].astype(str)
nn_linf_pct = ((pvt.loc['NN', 'ROARLInf'] - pvt.loc['NN', 'LInf']) * 100 / pvt.loc['NN', 'LInf']).round(2)
nn_linf_pct = nn_linf_pct.loc['mean'].astype(str)
nn_l1_pct = ((pvt.loc['NN', 'ROARL1'] - pvt.loc['NN', 'L1PSD']) * 100 / pvt.loc['NN', 'L1PSD']).round(2)
nn_l1_pct = nn_l1_pct.loc['mean'].astype(str)

In [122]:
# new_pvt.loc['LR', 'ROARLInf'] += " (" + lr_linf_pct + "%)"
# new_pvt.loc['LR', 'ROARL1'] += " (" + lr_l1_pct + "%)"
# new_pvt.loc['NN', 'ROARLInf'] += " (" + nn_linf_pct + "%)"
# new_pvt.loc['NN', 'ROARL1'] += " (" + nn_l1_pct + "%)"

new_pvt.loc['LR', 'ROARLInf'] = "(" + lr_linf_pct + "%) " + new_pvt.loc['LR', 'ROARLInf']
new_pvt.loc['LR', 'ROARL1'] = "(" + lr_l1_pct + "%) " + new_pvt.loc['LR', 'ROARL1']
new_pvt.loc['NN', 'ROARLInf'] = "(" + nn_linf_pct + "%) " + new_pvt.loc['NN', 'ROARLInf']
new_pvt.loc['NN', 'ROARL1'] = "(" + nn_l1_pct + "%) " + new_pvt.loc['NN', 'ROARL1']

In [123]:
new_pvt.style

In [124]:
means = pvt['mean']

mask = pd.DataFrame(False, index=new_pvt.index, columns=new_pvt.columns)
pairs = [('LInf', 'ROARLInf'), ('L1PSD', 'ROARL1')]

for model in means.index.get_level_values('Model').unique():
    for alg1, alg2 in pairs:
        s1 = means.loc[(model, alg1)]
        s2 = means.loc[(model, alg2)]

        metrics = s1.index.get_level_values('Metric').unique()

        for metric in metrics:
            match metric:
                case "J" | "Cost":
                    mask.loc[(model, alg1),(slice(None), slice(None), metric)] = s1.loc[:,:, metric] <= s2.loc[:,:, metric]
                    mask.loc[(model, alg2),(slice(None), slice(None), metric)] = s2.loc[:,:, metric] < s1.loc[:,:, metric]
                case "Current Validity" | "Worst Case Validity":
                    mask.loc[(model, alg1),(slice(None), slice(None), metric)] = s1.loc[:,:, metric] >= s2.loc[:,:, metric]
                    mask.loc[(model, alg2),(slice(None), slice(None), metric)] = s2.loc[:,:, metric] > s1.loc[:,:, metric]

                case _:
                    print(f"{metric} is not implemented")


# for model in means.index.get_level_values('Model').unique():
#     for a1, a2 in pairs:
#         if (model, a1) in means.index and (model, a2) in means.index:
#             s1 = means.loc[(model, a1)]   # row (Series over columns)
#             s2 = means.loc[(model, a2)]
#             # bold the lower value; bold both on ties
#             mask.loc[(model, a1)] = (s1 <= s2).reindex(mask.columns, fill_value=False)
#             mask.loc[(model, a2)] = (s2 <  s1).reindex(mask.columns, fill_value=False)

def bold_where(df):
    return pd.DataFrame(np.where(mask, 'font-weight:bold', ''), index=df.index, columns=df.columns)

# new_pvt_styled = new_pvt.copy(deep=True)
new_pvt_styled = (new_pvt.style.apply(bold_where, axis=None))
new_pvt_styled

# One Lambda All Metric

In [352]:
df_results_sampled = df_results[df_results['lambda'].isin([0.1])]
df_results_sampled

,model,dataset,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity,BCE Loss,J
18,lr,german,Alg1,0,0.1,0.1,4.044556,0.813284,0.713148,0.338066,0.742522
41,lr,german,Alg1,1,0.1,0.1,3.564100,0.677389,0.580620,0.546476,0.902886
64,lr,german,Alg1,2,0.1,0.1,4.692264,0.836227,0.719806,0.328774,0.798000
87,lr,german,Alg1,3,0.1,0.1,3.695547,0.824048,0.725373,0.321069,0.690624
110,lr,german,Alg1,4,0.1,0.1,4.649043,0.794955,0.677940,0.388697,0.853601
...,...,...,...,...,...,...,...,...,...,...,...
1352,nn,sba,ROARL1,0,0.1,0.1,3.310300,0.917970,0.902227,0.102929,0.433959
1364,nn,sba,ROARL1,1,0.1,0.1,3.630617,0.907155,0.889711,0.117017,0.480078
1376,nn,sba,ROARL1,2,0.1,0.1,3.917083,0.907788,0.891276,0.115186,0.506894
1388,nn,sba,ROARL1,3,0.1,0.1,5.465867,0.917390,0.899980,0.105452,0.652039


In [356]:
df_results_sampled['lambda'] = df_results_sampled['lambda'].astype(str)
df_results_sampled['model'] = df_results_sampled['model'].replace(['lr', 'nn'], ['LR', 'NN'])
df_results_sampled['dataset'] = df_results_sampled['dataset'].replace(['german', 'sba'], ['German', 'SBA'])

pvt = df_results_sampled.pivot_table(values=['Current Validity', 'Worst Case Validity','Cost','J'], index=['model', 'algorithm'], columns=['dataset'], aggfunc=["mean", "std"])

pvt = pvt.rename(index={'Alg1': 'LInf'}, level='algorithm')
pvt.index.set_names(["Model", "Algorithm"], inplace=True)
pvt.columns.set_names(['AggFunc', 'Metric', 'Dataset'], inplace=True)
pvt.columns = pvt.columns.reorder_levels(['AggFunc', 'Dataset', 'Metric'])
pvt = pvt.sort_index(axis=1)

lambda_order = ["0.1", "0.01"]
metric_order = ["Current Validity", "Worst Case Validity", "Cost", "J"]
algorithm_order = ['LInf', 'ROARLInf', 'L1PSD', 'ROARL1']
pvt = pvt.reindex(columns=metric_order, level= "Metric").reindex(index=algorithm_order, level="Algorithm")

pvt.style


In [359]:
new_pvt = pvt['mean'].round(4).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

new_pvt.style

In [360]:
lr_linf_pct = ((pvt.loc['LR', 'ROARLInf'] - pvt.loc['LR', 'LInf']) * 100 / pvt.loc['LR', 'LInf']).round(0).astype(int)
lr_linf_pct = lr_linf_pct.loc['mean'].astype(str)
lr_l1_pct = ((pvt.loc['LR', 'ROARL1'] - pvt.loc['LR', 'L1PSD']) * 100 / pvt.loc['LR', 'L1PSD']).round(0).astype(int)
lr_l1_pct = lr_l1_pct.loc['mean'].astype(str)
nn_linf_pct = ((pvt.loc['NN', 'ROARLInf'] - pvt.loc['NN', 'LInf']) * 100 / pvt.loc['NN', 'LInf']).round(0).astype(int)
nn_linf_pct = nn_linf_pct.loc['mean'].astype(str)
nn_l1_pct = ((pvt.loc['NN', 'ROARL1'] - pvt.loc['NN', 'L1PSD']) * 100 / pvt.loc['NN', 'L1PSD']).round(0).astype(int)
nn_l1_pct = nn_l1_pct.loc['mean'].astype(str)

new_pvt.loc['LR', 'ROARLInf'] += " (" + lr_linf_pct + "%)"
new_pvt.loc['LR', 'ROARL1'] += " (" + lr_l1_pct + "%)"
new_pvt.loc['NN', 'ROARLInf'] += " (" + nn_linf_pct + "%)"
new_pvt.loc['NN', 'ROARL1'] += " (" + nn_l1_pct + "%)"

new_pvt.style

# One Lambda - LInf vs L1

In [372]:
df_results_sampled = df_results[df_results['lambda'].isin([0.01]) & df_results['algorithm'].isin(['Alg1', 'L1PSD'])]
df_results_sampled

,model,dataset,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity,BCE Loss,J
9,lr,german,Alg1,0,0.1,0.01,11.535425,0.992060,0.971308,0.029111,0.144466
32,lr,german,Alg1,1,0.1,0.01,15.956258,0.989983,0.954210,0.046871,0.206434
55,lr,german,Alg1,2,0.1,0.01,11.986345,0.992976,0.971989,0.028411,0.148274
78,lr,german,Alg1,3,0.1,0.01,10.824120,0.992229,0.972529,0.027855,0.136096
101,lr,german,Alg1,4,0.1,0.01,13.212443,0.992274,0.967806,0.032723,0.164848
119,lr,german,L1PSD,0,0.1,0.01,9.887512,0.982088,0.974062,0.026280,0.125155
126,lr,german,L1PSD,1,0.1,0.01,13.736950,0.974823,0.955818,0.045188,0.182558
133,lr,german,L1PSD,2,0.1,0.01,10.075509,0.983539,0.975253,0.025058,0.125814
140,lr,german,L1PSD,3,0.1,0.01,9.124467,0.982960,0.975716,0.024584,0.115829
147,lr,german,L1PSD,4,0.1,0.01,11.479343,0.980024,0.969386,0.031093,0.145886


In [373]:
df_results_sampled['lambda'] = df_results_sampled['lambda'].astype(str)
df_results_sampled['model'] = df_results_sampled['model'].replace(['lr', 'nn'], ['LR', 'NN'])
df_results_sampled['dataset'] = df_results_sampled['dataset'].replace(['german', 'sba'], ['German', 'SBA'])

pvt = df_results_sampled.pivot_table(values=['Current Validity', 'Worst Case Validity','Cost','J'], index=['model', 'algorithm'], columns=['dataset'], aggfunc=["mean", "std"])

pvt = pvt.rename(index={'Alg1': 'LInf'}, level='algorithm')
pvt.index.set_names(["Model", "Algorithm"], inplace=True)
pvt.columns.set_names(['AggFunc', 'Metric', 'Dataset'], inplace=True)
pvt.columns = pvt.columns.reorder_levels(['AggFunc', 'Dataset', 'Metric'])
pvt = pvt.sort_index(axis=1)

metric_order = ["Current Validity", "Worst Case Validity", "Cost", "J"]
algorithm_order = ['L1PSD', 'LInf']
pvt = pvt.reindex(columns=metric_order, level= "Metric").reindex(index=algorithm_order, level="Algorithm")

pvt.style

In [374]:
new_pvt = pvt['mean'].round(4).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

new_pvt.style

In [375]:
new_pvt = pvt['mean'].round(4).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

lr_linf_pct = ((pvt.loc['LR', 'LInf'] - pvt.loc['LR', 'L1PSD']) * 100 / pvt.loc['LR', 'L1PSD']).round(0).astype(int)
lr_linf_pct = lr_linf_pct.loc['mean'].astype(str)
nn_linf_pct = ((pvt.loc['NN', 'LInf'] - pvt.loc['NN', 'L1PSD']) * 100 / pvt.loc['NN', 'L1PSD']).round(0).astype(int)
nn_linf_pct = nn_linf_pct.loc['mean'].astype(str)

new_pvt.loc['LR', 'LInf'] = "(" + lr_linf_pct + "%) " + new_pvt.loc['LR', 'LInf']
new_pvt.loc['NN', 'LInf'] = "(" + nn_linf_pct + "%) " + new_pvt.loc['NN', 'LInf']

new_pvt.style

# One Lambda - LInf vs L1 (Different Lambd by Dataset)

In [377]:
df_results_sampled = df_results[df_results['algorithm'].isin(['Alg1', 'L1PSD'])]
df_results_sampled

,model,dataset,algorithm,seed,alpha,lambda,Cost,Current Validity,Worst Case Validity,BCE Loss,J
0,lr,german,Alg1,0,0.1,0.1,4.044556,0.813284,0.713148,0.338066,0.742522
1,lr,german,Alg1,1,0.1,0.1,3.564100,0.677389,0.580620,0.546476,0.902886
2,lr,german,Alg1,2,0.1,0.1,4.692264,0.836227,0.719806,0.328774,0.798000
3,lr,german,Alg1,3,0.1,0.1,3.695547,0.824048,0.725373,0.321069,0.690624
4,lr,german,Alg1,4,0.1,0.1,4.649043,0.794955,0.677940,0.388697,0.853601
5,lr,german,L1PSD,0,0.1,0.1,3.834094,0.798330,0.770316,0.260986,0.644396
6,lr,german,L1PSD,1,0.1,0.1,3.866700,0.727446,0.682209,0.382457,0.769127
7,lr,german,L1PSD,2,0.1,0.1,4.277773,0.813215,0.775984,0.253682,0.681459
8,lr,german,L1PSD,3,0.1,0.1,3.464153,0.808592,0.782608,0.245123,0.591539
9,lr,german,L1PSD,4,0.1,0.1,4.172600,0.769223,0.727249,0.319043,0.736303


In [378]:
df_results_sampled['lambda'] = df_results_sampled['lambda'].astype(str)
df_results_sampled['model'] = df_results_sampled['model'].replace(['lr', 'nn'], ['LR', 'NN'])
df_results_sampled['dataset'] = df_results_sampled['dataset'].replace(['german', 'sba'], ['German', 'SBA'])

pvt = df_results_sampled.pivot_table(values=['Current Validity', 'Worst Case Validity','Cost','J'], index=['model', 'algorithm'], columns=['dataset'], aggfunc=["mean", "std"])

pvt = pvt.rename(index={'Alg1': 'LInf'}, level='algorithm')
pvt.index.set_names(["Model", "Algorithm"], inplace=True)
pvt.columns.set_names(['AggFunc', 'Metric', 'Dataset'], inplace=True)
pvt.columns = pvt.columns.reorder_levels(['AggFunc', 'Dataset', 'Metric'])
pvt = pvt.sort_index(axis=1)

metric_order = ["Current Validity", "Worst Case Validity", "Cost", "J"]
algorithm_order = ['L1PSD', 'LInf']
pvt = pvt.reindex(columns=metric_order, level= "Metric").reindex(index=algorithm_order, level="Algorithm")

pvt.style

In [379]:
new_pvt = pvt['mean'].round(4).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

new_pvt.style

In [380]:
new_pvt = pvt['mean'].round(4).applymap(lambda x: f"{x:.4f}").astype(str) + ' \u00B1 ' + \
    pvt['std'].round(2).applymap(lambda x: f"{x:.2f}").astype(str)

lr_linf_pct = ((pvt.loc['LR', 'LInf'] - pvt.loc['LR', 'L1PSD']) * 100 / pvt.loc['LR', 'L1PSD']).round(0).astype(int)
lr_linf_pct = lr_linf_pct.loc['mean'].astype(str)
nn_linf_pct = ((pvt.loc['NN', 'LInf'] - pvt.loc['NN', 'L1PSD']) * 100 / pvt.loc['NN', 'L1PSD']).round(0).astype(int)
nn_linf_pct = nn_linf_pct.loc['mean'].astype(str)

new_pvt.loc['LR', 'LInf'] = "(" + lr_linf_pct + "%) " + new_pvt.loc['LR', 'LInf']
new_pvt.loc['NN', 'LInf'] = "(" + nn_linf_pct + "%) " + new_pvt.loc['NN', 'LInf']

new_pvt.style